# SECOM Data Modeling
---

### Imports and creating test-train split

In [115]:
# Imports
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVC

In [100]:
# Pulling in the SECOM data and loading data into feature and label dataframes
secom = fetch_ucirepo(id=179)
df = pd.DataFrame(secom.data.original)
X = df.drop(columns=["class", "timestamp"])
y = df["class"]
# converting "-1" passing label to "0"
y = y.replace(-1, 0)

In [101]:
# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [102]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

class
0    0.933759
1    0.066241
Name: proportion, dtype: float64
class
0    0.933121
1    0.066879
Name: proportion, dtype: float64


___

### Creating custom Secom dataset preprocess transformer

In [103]:
class SecomPreProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing_threshold: int, corr_threshold: int):
        self.missing_threshold = missing_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X: pd.DataFrame, y=None):
        X = X.copy()

        # filtering out features with null percentage above threshold
        self.null_cols_ = X.columns[X.isna().mean() > self.missing_threshold].to_list()
        X = X.drop(columns=self.null_cols_)
        
        # imputing NaN with median
        self.medians_ = X.median()
        X = X.fillna(self.medians_)

        # Remove constant variance features
        self.zero_var_cols_ = X.columns[X.var() == 0].to_list()
        X = X.drop(columns=self.zero_var_cols_)

        # dropping columns correlated above threshold
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        self.corr_cols_ = [
            col
            for col in upper.columns
            if any(upper[col] > self.corr_threshold)
        ]

        X = X.drop(columns=self.corr_cols_)

        self.feature_names_out_ = X.columns.to_list()

        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        X = X.reindex(columns=self.feature_names_out_)

        X = X.fillna(self.medians_)

        return X

    def get_feature_names_out(self, input_features=None) -> np.array:
        return np.array(self.feature_names_out_)

___

### Custom PLSTransformer 
Needed since it returns X and y arrays, where models such as logistic regression expect only X.
This PLSTransformer is simply a custom transformer wrapper around PLSRegression to capture the X array of latent variables to be used in subsequent modeling.

In [104]:
# Creating custom transformer to extract PLS componenets
class PLSTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=10):
        self.n_components = n_components

    def fit(self, X, y):
        self.pls_ = PLSRegression(n_components=self.n_components)
        self.pls_.fit(X, y)
        return self

    def transform(self, X):
        return self.pls_.transform(X)

___

### Creating a loop to test the performance of several models at once

In [105]:
# specifying dictionaries with two categories of models: ones that need scaling and those that do not (typically tree-based models)
models_needs_scaling = {
    "Logistic Regression": LogisticRegression(
        penalty='l1', 
        random_state=42, 
        class_weight="balanced", 
        solver='liblinear', 
        max_iter=5000        
    ),
    "SVM Linear": SVC(
        kernel="linear",
        C=1.0,
        class_weight="balanced",
        probability=True,
        random_state=42
    ),
    "SVM RBF": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=42
    )
}
models_no_scaling = {
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Random Forest (small)": RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10,
        min_samples_split=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Histogram Gradient Boost": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=4,
        max_iter=300,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42,
        eval_metric="logloss"
    )
}

In [106]:
# Adding this custom pipeline dictionary as the PLS modeling is a special case as it needs the special PLSTransformer we created
models_custom_pipelines = {
    f"PLS-DA {n} components": Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("scale", StandardScaler()),
        ("pls", PLSTransformer(n_components=n)),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    ])
    for n in [2, 5, 10, 15, 20, 30]
}

In [109]:
# creating a function to run cross validation on multiple models at once and storing otputting their performance to a dataframe
def run_performance(models_needs_scaling: dict, models_no_scaling: dict, models_custom_pipelines: dict | None = None, 
                    X_train : pd.DataFrame = X_train, y_train: pd.DataFrame = y_train) -> pd.DataFrame:
    results = []

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    def evaluate_pipe(name, pipe):
        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "pr_auc": "average_precision"
            },
            return_train_score=True,
            n_jobs=-1
        )

        results.append({
            "model": name,
            "train_roc_auc": scores["train_roc_auc"].mean(),
            "train_pr_auc": scores["train_pr_auc"].mean(),
            "cv_roc_auc_mean": scores["test_roc_auc"].mean(),
            "cv_roc_auc_std": scores["test_roc_auc"].std(),
            "cv_pr_auc_mean": scores["test_pr_auc"].mean(),
            "cv_pr_auc_std": scores["test_pr_auc"].std(),
        })


    for name, model in models_needs_scaling.items():
        pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("scale", StandardScaler()),
        ("model", model)
        ])

        evaluate_pipe(name, pipe)
        

    for name, model in models_no_scaling.items():
        pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("model", model)
        ])

        evaluate_pipe(name, pipe)

    if models_custom_pipelines is not None:
        for name, pipe in models_custom_pipelines.items():
            evaluate_pipe(name, pipe)

    performance_summary = pd.DataFrame(results).sort_values("cv_pr_auc_mean", ascending=False).reset_index(drop=True)
    return performance_summary

In [110]:
# Running performance checks
performance_summary = run_performance(models_needs_scaling=models_needs_scaling, models_no_scaling=models_no_scaling, 
                                      models_custom_pipelines=models_custom_pipelines)
performance_summary

,model,train_roc_auc,train_pr_auc,cv_roc_auc_mean,cv_roc_auc_std,cv_pr_auc_mean,cv_pr_auc_std
0,Random Forest (small),1.000000,1.000000,0.724020,0.048940,0.200041,0.052828
1,Random Forest,1.000000,1.000000,0.722766,0.054694,0.198035,0.059456
2,XGBoost,1.000000,1.000000,0.698303,0.063096,0.194247,0.048157
3,PLS-DA 2 components,0.901980,0.529548,0.662689,0.058279,0.161826,0.057569
4,PLS-DA 5 components,0.944674,0.702331,0.646732,0.037519,0.153672,0.052328
5,Logistic Regression,0.998701,0.970597,0.599604,0.082212,0.153012,0.079866
6,SVM Linear,0.999822,0.994890,0.577256,0.077944,0.148724,0.075724
7,SVM RBF,1.000000,1.000000,0.654456,0.083712,0.145575,0.051439
8,PLS-DA 20 components,0.980379,0.836153,0.646952,0.045097,0.143164,0.036943
9,Histogram Gradient Boost,1.000000,1.000000,0.655505,0.061309,0.142040,0.017916


___

### Performing hyperparameter tuning on the best model

In [111]:
# Creating a pipe with the best model from the model selection step
rf_pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10,
        min_samples_split=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
        ))
])

In [112]:
# Creating the hyperparamter tuning dictionary with params and values to randomly search
param_dist = {
    "model__max_depth": [4, 6, 8, 10, 12, None],
    "model__min_samples_leaf": [2, 5, 10, 20, 30],
    "model__min_samples_split": [5, 10, 20, 40, 80],
    "model__max_features": ["sqrt", 0.10, 0.25, 0.50]
}

search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",
    cv=5,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [4, 6, ...], 'model__max_features': ['sqrt', 0.1, ...], 'model__min_samples_leaf': [2, 5, ...], 'model__min_samples_split': [5, 10, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``

In [113]:
# Extracting the best pr-auc and hyperparameters from the random search
best_params_df = pd.DataFrame([search.best_params_])
best_params_df["cv_pr_auc"] = round(search.best_score_, 3)

In [114]:
# Running predictions using the best pipeline on the holdout test set to determine performance
best_pipe = search.best_estimator_
y_proba = best_pipe.predict_proba(X_test)[:, 1]
best_params_df["test_pr_auc"] = round(average_precision_score(y_test, y_proba),3)
best_params_df["test_roc_auc"] = round(roc_auc_score(y_test, y_proba),3)
best_params_df

,model__min_samples_split,model__min_samples_leaf,model__max_features,model__max_depth,cv_pr_auc,test_pr_auc,test_roc_auc
0,20,10,0.1,12,0.236,0.214,0.759


___

### Fitting the best tuned model on all the training data and running a threshold sweep to identify the optimum cutoff for failed wafers

In [127]:
def threshold_sweep_from_proba(y_true, y_proba, thresholds):
    baseline_failure_rate = y_true.mean()
    results = []

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)

        results.append({
            "threshold": threshold,
            "precision": round(precision, 2),
            "recall": round(recall_score(y_true, y_pred), 2),
            "f1": round(f1_score(y_true, y_pred), 2),
            "flagged": y_pred.sum(),
            "flagged_rate": round(y_pred.mean(), 2),
            "enrichment": round(precision / baseline_failure_rate, 2)
        })

    return (
        pd.DataFrame(results)
        .sort_values("threshold", ascending=False)
        .reset_index(drop=True)
    )

In [128]:
def eval_threshold(best_pipe: Pipeline, thresholds: list[int], X_train: pd.DataFrame = X_train, 
                   y_train: pd.DataFrame = y_train) -> pd.DataFrame:
    cv_proba = cross_val_predict(
        best_pipe,
        X_train,
        y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    return threshold_sweep_from_proba(
        y_true=y_train,
        y_proba=cv_proba,
        thresholds=thresholds
    )

In [129]:
thresholds = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
cv_threshold_df = eval_threshold(best_pipe=best_pipe, thresholds=thresholds)
cv_threshold_df

,threshold,precision,recall,f1,flagged,flagged_rate,enrichment
0,0.50,0.23,0.12,0.16,43,0.03,3.51
1,0.40,0.19,0.36,0.25,157,0.13,2.88
2,0.30,0.13,0.59,0.22,369,0.29,2.00
3,0.20,0.09,0.86,0.16,798,0.64,1.34
4,0.10,0.07,1.00,0.13,1224,0.98,1.02
5,0.05,0.07,1.00,0.12,1253,1.00,1.00


In [130]:
# Selected threshold for max performance
selected_threshold = cv_threshold_df.loc[cv_threshold_df["f1"].idxmax(), "threshold"]

In [131]:
# Final model performance evaluation
def eval_test_final(best_pipe: Pipeline, selected_threshold: int, X_train: pd.DataFrame = X_train, y_train: pd.DataFrame = y_train,
                    X_test: pd.DataFrame = X_test, y_test: pd.DataFrame = y_test):
    
    best_pipe.fit(X_train, y_train)
    test_proba = best_pipe.predict_proba(X_test)[:, 1]

    return threshold_sweep_from_proba(
        y_true=y_test,
        y_proba=test_proba,
        thresholds=[selected_threshold]
    )

In [132]:
# Run final model performance evaluation
eval_test_final_df = eval_test_final(best_pipe=best_pipe, selected_threshold=selected_threshold)
eval_test_final_df

,threshold,precision,recall,f1,flagged,flagged_rate,enrichment
0,0.4,0.26,0.38,0.31,31,0.1,3.86


___

### Assessing feature importance in the best model

In [ ]:
feature_importances = best_pipe.named_steps["model"].feature_importances_
feature_names = best_pipe.named_steps["preprocess"].get_feature_names_out()

feature_importance_df = pd.DataFrame({
    "sensor": feature_names,
    "feature_importance": feature_importances
    })

feature_importance_df.sort_values("feature_importance", ascending=False).head(20)

In [ ]:
# perm = permutation_importance(
#     best_pipe,
#     X_test,
#     y_test,
#     scoring="average_precision",
#     n_repeats=20,
#     random_state=42,
#     n_jobs=-1
# )

In [ ]:
# perm_df = pd.DataFrame({
#     "feature": X_test.columns,
#     "importance_mean": perm.importances_mean,
#     "importance_std": perm.importances_std
# }).sort_values("importance_mean", ascending=False)

# perm_df.head(20)